In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
INDEXED_OLIGO_DATA = 'aso_inhibitions_with_canonical_gene_indexed.csv'
OLIGO_DATA = 'aso_inhibitions_with_canonical_gene.csv'

In [3]:
import pandas as pd

data = pd.read_csv('aso_inhibitions_with_canonical_gene.csv')

In [4]:
# Check if the column exists
if 'index_oligo' not in data.columns:
    # Create the column starting at 1
    data['index_oligo'] = range(1, len(data) + 1)

    # Save the file
    data.to_csv('aso_inhibitions_with_canonical_gene_indexed.csv', index=False)
    print("Added 'index_oligo' and saved file.")
else:
    print("'index_oligo' already exists. No changes made.")

Added 'index_oligo' and saved file.


In [5]:
from tauso.data.consts import SEQUENCE, CELL_LINE, CANONICAL_GENE, CELL_LINE_ORGANISM, INHIBITION, VOLUME

rename_scheme = {'aso_sequence_5_to_3': SEQUENCE, 'Canonical Gene Name': CANONICAL_GENE, 'cell_line': CELL_LINE,
                 'cell_line_species': CELL_LINE_ORGANISM, 'inhibition_percent': INHIBITION, 'dosage': VOLUME}
data = data.rename(columns=rename_scheme)

In [6]:
data = data[data['steric_blocking'] == False]
data = data[~data[CANONICAL_GENE].str.contains(';')]
data = data[data[INHIBITION].notna()]

In [7]:
from tauso.data.consts import CELL_LINE_TO_DEPMAP_PROXY_DICT, CELL_LINE_DEPMAP, CELL_LINE_DEPMAP_PROXY, \
    CELL_LINE_TO_DEPMAP

data[CELL_LINE_DEPMAP_PROXY] = data[CELL_LINE].map(CELL_LINE_TO_DEPMAP_PROXY_DICT)
data[CELL_LINE_DEPMAP] = data[CELL_LINE_DEPMAP_PROXY].map(CELL_LINE_TO_DEPMAP)

In [8]:
# Create the binary columns
binary_features = pd.get_dummies(data['transfection_method'])

# Join them back to your original dataframe (optional)
data = pd.concat([data, binary_features], axis=1)

# Display the result
print(data[['transfection_method'] + list(binary_features.columns)].head())

  transfection_method  Electroporation  Gymnosis  Lipofection  Other
0     Electroporation             True     False        False  False
1     Electroporation             True     False        False  False
2     Electroporation             True     False        False  False
3     Electroporation             True     False        False  False
4     Electroporation             True     False        False  False


In [9]:
from notebooks.features.feature_extraction import save_feature

TRANSFECTION_FEATURES = binary_features.columns.tolist()
for feature in ['Electroporation', 'Gymnosis', 'Lipofection', 'Other']:
    save_feature(data, feature, version='oligo')

File exists for 'Electroporation' but values are identical (within tolerance). No action taken.
File exists for 'Gymnosis' but values are identical (within tolerance). No action taken.
File exists for 'Lipofection' but values are identical (within tolerance). No action taken.
File exists for 'Other' but values are identical (within tolerance). No action taken.


In [10]:
target_genes = data[CANONICAL_GENE].unique().tolist()

In [11]:
target_genes.append('RNASEH1')

In [12]:
from tauso.genome.read_human_genome import get_locus_to_data_dict

gene_to_data = get_locus_to_data_dict(include_introns=True,
                                      gene_subset=target_genes)  # Get all genes, we will use later

[Task] finished in 0.0028s
Elapsed DB:  0.002783060073852539
Elapsed Fasta:  0.002783060073852539
Length:  3267117988


In [13]:
from tauso.populate.populate_structure import get_populated_df_with_structure_features

data = get_populated_df_with_structure_features(data, target_genes, gene_to_data)

In [ ]:
for feature in ['sense_start', 'sense_start_from_end', 'sense_length', 'sense_exon', 'sense_intron', 'sense_utr',
                'sense_type']:
    save_feature(data, feature, version='oligo')

In [27]:
import os
from tauso.data.data import get_data_dir
from tauso.data.consts import CELL_LINE_DEPMAP
from tauso.features.codon_usage.find_cai_reference import load_cell_line_gene_expression

data_dir = get_data_dir()
expression_dir = os.path.join(data_dir, "processed_expression")
depmap_ids = list(set(data[CELL_LINE_DEPMAP]))

# 2. Run the Optimized Loader
#    This loads the DB/FASTA once, filters for valid genes, and enriches sequences.
transcriptomes = load_cell_line_gene_expression(
    depmap_ids,
    target_genes,
    expression_dir=expression_dir,  # Directory containing expression CSVs
)

  -> Warning: No expression data for ACH-002310
  -> Warning: No expression data for ACH-001087
  -> Warning: No expression data for nan


In [25]:
from tauso.features.hybridization_off_target.common import get_general_expression_of_genes
from tauso.data.data import get_data_dir
from pathlib import Path

mean_exp_data = get_general_expression_of_genes(
    Path(get_data_dir()) / 'OmicsExpressionTPMLogp1HumanAllGenesStranded.csv', target_genes)
transcriptomes['general'] = mean_exp_data

Filtering: Kept 289 genes out of 41144 total columns.


In [28]:
from tauso.populate.populate_context import populate_mrna_expression

data, expression_features = populate_mrna_expression(data, transcriptomes)

In [29]:
for feature in expression_features:
    save_feature(data, feature, version='oligo')

Saved feature: target_expression
File exists for 'rnase_expression' but values are identical (within tolerance). No action taken.


In [ ]:
from tauso.populate.populate_rnase import populate_rnase_features

data, rnase_features = populate_rnase_features(data)

In [ ]:
for feature in rnase_features:
    save_feature(data, feature, version='oligo', overwrite=True)

In [ ]:
from tauso.features.hybridization_off_target.RNaseH1_off_target import off_target_specific_seq_pandarallel
from notebooks.features.feature_extraction import save_feature

In [ ]:
from tauso.features.hybridization_off_target.RNaseH1_off_target import on_target_total_hybridization

off_target_features = []
cutoffs = [0, 1200]

jobs = 32

for cutoff in cutoffs:
    data, feature_name = off_target_specific_seq_pandarallel(data, 'RNASEH1', gene_to_data, cutoff=cutoff, n_jobs=jobs,
                                                             verbose=True)
    save_feature(data, feature_name, version='oligo')
    off_target_features.append(feature_name)

    data, feature_name = off_target_specific_seq_pandarallel(data, 'ACTB', gene_to_data, cutoff=cutoff, n_jobs=jobs,
                                                             verbose=True)
    save_feature(data, feature_name, version='oligo')
    off_target_features.append(feature_name)

    data, feature_name = on_target_total_hybridization(data, gene_to_data, cutoff=cutoff, n_jobs=jobs,
                                                       verbose=True)
    save_feature(data, feature_name, version='oligo')
    off_target_features.append(feature_name)

In [ ]:
from tauso.populate.populate_fold import populate_mfe_features

data, mfe_features = populate_mfe_features(data, gene_to_data, n_jobs=32)

In [ ]:
from notebooks.features.feature_extraction import save_feature

In [ ]:
for feature in mfe_features:
    save_feature(data, feature, version='oligo')

In [ ]:
from tauso.populate.populate_fold import populate_sense_accessibility_batch

configurations = [
    {"flank": 120, "access": 20, "seeds": [13]},
    {"flank": 120, "access": 13, "seeds": [4, 6, 8]},
    {"flank": 120, "access": 20, "seeds": [4, 6, 8]},
    {"flank": 120, "access": 13, "seeds": [13, 26, 39]}
]

feature_names = []
for config in configurations:
    c_flank = config["flank"]
    c_access = config["access"]
    c_seeds = config["seeds"]

    print(f"Running: Flank={c_flank}, Access={c_access}, Seeds={c_seeds})...")

    data, feature_name = populate_sense_accessibility_batch(
        data,
        gene_to_data,
        batch_size=1000,
        flank_size=c_flank,
        access_size=c_access,
        seed_sizes=c_seeds,
        n_jobs=16
    )
    try:
        save_feature(df=data, feature_name=feature_name, version='oligo')
        print(f"Saved: {feature_name}")
    except Exception as e:
        print("Error: ", e)
        pass

In [ ]:
from tauso.populate.populate_sequence import populate_sequence_features

data, sequence_features = populate_sequence_features(data, cpus=32)

In [ ]:
for feature in sequence_features:
    save_feature(data, feature, version='oligo')

In [ ]:
from tauso.populate.populate_sequence import populate_sequence_one_hot_encoded

data, sequence_one_hot_features = populate_sequence_one_hot_encoded(data, cpus=32)

In [ ]:
for feature in sequence_one_hot_features:
    save_feature(data, feature, version='oligo')

In [ ]:
for feature in sequence_features:
    save_feature(data, feature, version='oligo')

In [ ]:
from tauso.populate.populate_sequence_chemistry import populate_sequence_chemistry_features

data, features = populate_sequence_chemistry_features(data)

In [ ]:
for feature in features:
    save_feature(data, feature, version='oligo')

In [30]:
from notebooks.competitors.oligo_ai.parse_chemistry import populate_chemistry

data = populate_chemistry(data)

In [32]:
from tauso.data.consts import PS_PATTERN

data['is_all_ps'] = data[PS_PATTERN].apply(lambda x: 1 if x and x.replace('*', '') == '' else 0)

In [ ]:
# 2. Max Consecutive PO: Find the longest contiguous string of 'd's
data['max_consecutive_PO'] = data[PS_PATTERN].apply(
    lambda x: max((len(chunk) for chunk in x.split('*')), default=0) if isinstance(x, str) else 0
)

In [ ]:
import re

data['ps_end_score'] = data[PS_PATTERN].apply(
    lambda x: len(re.match(r'^\**', x).group()) + len(re.search(r'\**$', x).group()) if isinstance(x, str) else 0
)


In [ ]:
data['total_po'] = data[PS_PATTERN].apply(lambda x: x.count('d') if isinstance(x, str) else 0)
data['total_ps'] = data[PS_PATTERN].apply(lambda x: x.count('*') if isinstance(x, str) else 0)
data['backbone_length'] = data['total_po'] + data['total_ps'] # ASO length - 1


data['po_percentage'] = data.apply(
    lambda row: row['total_po'] / row['backbone_length'] if row['backbone_length'] > 0 else 0, axis=1
)

In [33]:
from notebooks.features.feature_extraction import save_feature

save_feature(data, 'is_all_ps', version='oligo')
save_feature(data, 'po_percentage', version='oligo')
save_feature(data, 'ps_end_score', version='oligo')
save_feature(data, 'max_consecutive_PO', version='oligo')

File exists for 'is_all_ps' but values are identical (within tolerance). No action taken.


KeyError: "['po_percentage'] not in index"

In [ ]:
from tauso.populate.populate_hybridization import populate_hybridization

data, hybridization_features = populate_hybridization(data, n_cores=32)

In [ ]:
for feature in hybridization_features:
    save_feature(data, feature, version='oligo')

In [ ]:
from tauso.populate.populate_modification import populate_modifications

data, modification_features = populate_modifications(data, n_cores=32)

In [ ]:
for feature in modification_features:
    save_feature(data, feature, version='oligo')

In [ ]:
from tauso.data.data import get_paths
from tauso.genome.TranscriptMapper import GeneCoordinateMapper
from tauso.features.context.ribo_seq import add_genomic_coordinates

paths = get_paths('GRCh38')
mapper = GeneCoordinateMapper(paths['db'])
new = add_genomic_coordinates(data, mapper)

In [ ]:
from tauso.populate.populate_context import populate_ribo_seq

data, ribo_features = populate_ribo_seq('human', new)

In [ ]:
for feature in ribo_features:
    save_feature(data, feature, version='oligo')

In [19]:
from tauso.genome.TranscriptMapper import GeneCoordinateMapper, build_gene_sequence_registry
from tauso.data.data import get_paths

paths = get_paths('GRCh38')
mapper = GeneCoordinateMapper(paths['db'])

# D. Build the Registry object
# This provides the {gene: {'cds_sequence': '...'}} mapping
ref_registry = build_gene_sequence_registry(
    genes=target_genes,
    gene_to_data=gene_to_data,
    mapper=mapper
)

print(f"Registry rebuilt for {len(ref_registry)} experimental genes.")

Registry rebuilt for 291 experimental genes.


In [20]:
from tauso.algorithms.genomic_context_windows import add_external_mrna_and_context_columns

FLANK_SIZES_PREMRNA = [20, 30, 40, 50, 60, 70]
CDS_WINDOWS = [20, 30, 40, 50, 60, 70]

# Run the optimized context generator
data = add_external_mrna_and_context_columns(
    df=data,
    mapper=mapper,
    gene_registry=ref_registry,
    flank_sizes_premrna=FLANK_SIZES_PREMRNA,
    flank_sizes_cds=CDS_WINDOWS
)

print("Genomic context added.")

Genomic context added.


In [ ]:
from tauso.populate.populate_codon_usage import populate_tai

data, tai_features = populate_tai(data, CDS_WINDOWS, ref_registry)

In [ ]:
for feature in tai_features:
    save_feature(data, feature, version='oligo')

In [ ]:
from tauso.populate.populate_codon_usage import populate_cai

data, cai_features = populate_cai(data, CDS_WINDOWS, ref_registry)

In [ ]:
for feature in cai_features:
    save_feature(data, feature, version='oligo', overwrite=True)

In [ ]:
from tauso.populate.populate_codon_usage import populate_enc

data, enc_features = populate_enc(data, CDS_WINDOWS, ref_registry, n_jobs=32)

In [ ]:
for feature in enc_features:
    save_feature(data, feature, version='oligo')

In [ ]:
from tauso.data.data import load_db

db = load_db()

In [16]:
from tauso.genome.read_human_genome import get_locus_to_data_dict

gene_to_data = get_locus_to_data_dict(include_introns=True)


[Task] finished in 0.0005s
Elapsed DB:  0.0005183219909667969
Elapsed Fasta:  0.0005183219909667969
Length:  3267117988


In [ ]:
from tauso.common.gtf import filter_gtf_genes
import os
from tauso.data.consts import CELL_LINE_DEPMAP
from tauso.features.codon_usage.find_cai_reference import load_cell_line_gene_expression

from tauso.features.hybridization_off_target.common import get_general_expression_of_genes
from tauso.data.data import get_data_dir
from pathlib import Path

data_dir = get_data_dir()
expression_dir = os.path.join(data_dir, "processed_expression")
depmap_ids = list(set(data[CELL_LINE_DEPMAP]))

valid_genes = filter_gtf_genes(db, filter_mode='non_mt')

# 2. Run the Optimized Loader
#    This loads the DB/FASTA once, filters for valid genes, and enriches sequences.
transcriptomes = load_cell_line_gene_expression(
    depmap_ids,
    valid_genes,
    expression_dir=expression_dir,  # Directory containing expression CSVs
)

mean_exp_data = get_general_expression_of_genes(
    Path(get_data_dir()) / 'OmicsExpressionTPMLogp1HumanAllGenesStranded.csv', valid_genes)
transcriptomes['general'] = mean_exp_data

In [ ]:
from tauso.features.hybridization_off_target.add_off_target_feat import AggregationMethod
from tauso.features.hybridization_off_target.off_target_feature import populate_off_target_general, \
    populate_off_target_specific

top_n_list = [50, 100, 200]
cutoff_list = [800, 1000, 1200]

for n in top_n_list:
    for cutoff in cutoff_list:
        data, off_target_general_features = populate_off_target_general(
            ASO_df=data,
            gene_to_data=gene_to_data,
            cell_line2data=transcriptomes,
            top_n_list=[n],
            cutoff_list=[cutoff],
            method=AggregationMethod.ARTM,
            n_cores=32)
        for feature in off_target_general_features:
            save_feature(data, feature, version='oligo', overwrite=True)

In [ ]:
from tauso.features.hybridization_off_target.add_off_target_feat import AggregationMethod
from tauso.features.hybridization_off_target.off_target_feature import populate_off_target_general

top_n_list = [200]
cutoff_list = [1000, 1200]

for n in top_n_list:
    for cutoff in cutoff_list:
        data, off_target_general_features = populate_off_target_general(
            ASO_df=data,
            gene_to_data=gene_to_data,
            cell_line2data=transcriptomes,
            top_n_list=[n],
            cutoff_list=[cutoff],
            method=AggregationMethod.MECH,
            n_cores=24)
        for feature in off_target_general_features:
            save_feature(data, feature, version='oligo', overwrite=True)

In [ ]:
from tauso.features.hybridization_off_target.add_off_target_feat import AggregationMethod
from tauso.features.hybridization_off_target.off_target_feature import populate_off_target_general, \
    populate_off_target_specific

top_n_list = [25, 50, 100, 200]
cutoff_list = [800, 1000, 1200]

for n in top_n_list:
    for cutoff in cutoff_list:
        data, off_target_general_features = populate_off_target_specific(
            ASO_df=data,
            gene_to_data=gene_to_data,
            cell_line2data=transcriptomes,
            top_n_list=[n],
            cutoff_list=[cutoff],
            method=AggregationMethod.ARTM,
            n_cores=24)
        for feature in off_target_general_features:
            save_feature(data, feature, version='oligo', overwrite=True)

In [14]:
from tauso.features.rbp.load_rbp import load_attract_data

rbp_map, pwm_db = load_attract_data()

Loading RBP metadata from /home/michael/.local/share/tauso/RBS_motifs_Homo_sapiens.csv...
Loading PWM matrices from /home/michael/.local/share/tauso/pwm.txt...
Loaded 1583 PWMs covering 160 unique RBPs.


In [15]:
from tauso.features.rbp.pwm_helper import build_rbp_expression_matrix

expression_matrix = build_rbp_expression_matrix(df=data, pwm_db=pwm_db)

Loading metadata from /home/michael/.local/share/tauso/RBS_motifs_Homo_sapiens.csv to map Matrix IDs to Gene Names...
Mapped 1583 matrices to 160 unique gene names.
Loading expression for 25 cell lines...
  -> Warning: No expression data for nan
  -> Warning: No expression data for ACH-002310
  -> Warning: No expression data for ACH-001087
Building lookup matrix...
Success. Expression matrix ready: (22, 158)


In [21]:
from tauso.populate.populate_rbp import populate_rbp_interaction_features, populate_complexity_features
from notebooks.features.feature_extraction import save_feature

for flank_size in [50]:
    window_col = f'flank_sequence_{flank_size}'

    data, individual_features = populate_rbp_interaction_features(
        data, rbp_map, pwm_db, expression_matrix, gene_to_data,
        window_col, n_jobs=32
    )

    data, global_features = populate_complexity_features(
        data,
        individual_features,
        suffix=str(flank_size), type='expression'
    )
print("Done.")

22 cells, 158 valid RBPs found.
Calculating interaction features for 158 RBPs on 158725 rows...


Computing Batches: 100%|██████████| 65/65 [00:07<00:00,  9.28it/s] 


Assigning columns...


/home/michael/career/tauso_article/tauso_source2/src/tauso/populate/populate_rbp.py:309: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = values
/home/michael/career/tauso_article/tauso_source2/src/tauso/populate/populate_rbp.py:309: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = values
/home/michael/career/tauso_article/tauso_source2/src/tauso/populate/populate_rbp.py:309: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

Done. Added 158 interaction features.


/home/michael/career/tauso_article/tauso_source2/src/tauso/populate/populate_rbp.py:389: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[total_col] = total_scores


Done.


/home/michael/career/tauso_article/tauso_source2/src/tauso/populate/populate_rbp.py:401: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[div_col] = entropy(probs, axis=1)


In [23]:
for feature in individual_features:
    save_feature(data, feature, version='oligo', overwrite=True)

# # Save global features
for feature in global_features:
    save_feature(data, feature, version='oligo', overwrite=True)

print("Done.")

!!! Conflict detected for feature: RBP_A1CF_expr_aff_50 !!!
Found 17634 differing rows. Top 10 differences:
     index_oligo  RBP_A1CF_expr_aff_50_new  RBP_A1CF_expr_aff_50_existing
595          596                       NaN                            0.0
596          597                       NaN                            0.0
597          598                       NaN                            0.0
598          599                       NaN                            0.0
599          600                       NaN                            0.0
600          601                       NaN                            0.0
601          602                       NaN                            0.0
602          603                       NaN                            0.0
603          604                       NaN                            0.0
604          605                       NaN                            0.0
------------------------------
Overwrote feature: RBP_A1CF_expr_aff_50
Overwro

In [ ]:
from tauso.populate.populate_rbp import populate_rbp_affinity_features, populate_complexity_features
from notebooks.features.feature_extraction import save_feature

for flank_size in [50]:
    window_col = f'flank_sequence_{flank_size}'

    data, individual_features = populate_rbp_affinity_features(
        data, rbp_map, pwm_db, gene_to_data,
        window_col, n_jobs=32
    )

    data, global_features = populate_complexity_features(
        data,
        individual_features,
        suffix=str(flank_size), type='generic'
    )
print("Done.")

In [ ]:
for feature in individual_features:
    save_feature(data, feature, version='oligo', overwrite=True)

# Save global features
for feature in global_features:
    save_feature(data, feature, version='oligo')

print("Done.")

In [ ]:
from tauso.populate.populate_rbp import populate_functional_features
from tauso.features.rbp.rbp_annotations import rbp_role_map_strict

print("Calculating Stabilizer/Destabilizer features (Strict Map)...")

data, functional_features = populate_functional_features(
    data,
    rbp_role_map_strict,
    flank_size=50
)

In [ ]:
# Save the new features
print(f"Saving {len(functional_features)} functional features...")
for feature in functional_features:
    save_feature(data, feature, version='oligo', overwrite=True)

In [ ]:
from tauso.populate.populate_rbp import populate_rbp_region
from tauso.features.rbp.RBP_features import create_positional_sequence_columns
import warnings
from pandas.errors import PerformanceWarning

# 1. SILENCE WARNINGS
warnings.simplefilter(action='ignore', category=PerformanceWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# 2. SLICE SEQUENCES (Once)
print("Slicing sequences...")
data = create_positional_sequence_columns(data, "flank_sequence_50", flank_size=50)

features = []
# 3. LOOP REGIONS
for region in ['left', 'core', 'right']:
    # Fix Memory Fragmentation
    data = data.copy()

    # --- CALL THE FUNCTION ---
    data, new_features = populate_rbp_region(
        df=data,
        region_name=region,
        rbp_map=rbp_map,
        pwm_db=pwm_db,
        expr_matrix=expression_matrix,
        role_map=rbp_role_map_strict,
        gene_to_data=gene_to_data,
    )
    features.extend(new_features)

print("Done.")

In [ ]:
# 4. SAVE IMMEDIATELY
print(f"Saving {len(new_features)} features for {region}...")
for feature in new_features:
    save_feature(data, feature, version='oligo', overwrite=True)

In [ ]:
data[CELL_LINE].unique()

In [ ]:
from tauso.features.context.mrna_halflife import populate_mrna_halflife_features, load_halflife_mapping, \
    HalfLifeProvider

print("--- Initializing mRNA Half-Life Oracle ---")

# 1. Load the data (Run once within function scope)
mapping = load_halflife_mapping()
provider = HalfLifeProvider(mapping)

data, features = populate_mrna_halflife_features(data, provider)

In [ ]:
for feature in features:
    save_feature(data, feature, version='oligo', overwrite=True)

In [ ]:
transfection_features = ['Electroporation', 'Gymnosis', 'Lipofection', 'Other']

In [ ]:
all_features = sequence_features + list(hybridization_features.keys()) + list(modification_features.keys()) + [
    'sense_length', 'sense_start', 'sense_exon', 'sense_utr', 'sense_intron',
    'sense_start_from_end'] + ribo_features + tai_features + enc_features + mfe_features + transfection_features + [
                   'rnase_expression'] + ['target_expression']